# Desafio: detecção de anomalias em ECG com LSTM Autoencoder

Neste desafio você vai **completar o código** de um autoencoder recorrente (LSTM) que detecta batimentos cardíacos anômalos. O notebook é baseado no tutorial *Time Series Anomaly Detection using LSTM Autoencoders with PyTorch* (Venelin Valkov), com adaptações para tornar o código mais rápido e compatível com versões recentes do pandas.

Os trechos marcados com `# TODO` precisam ser completados. Após cada trecho há **verificações automáticas** (`assert`) para você conferir se acertou.

## Ideia

1. Treinamos o autoencoder **apenas com batimentos normais**.
2. O modelo aprende a reconstruir bem o que é normal.
3. Um batimento anômalo (nunca visto) será reconstruído **mal**: o erro de reconstrução fica alto.
4. Escolhemos um **limiar** de erro: acima dele, o batimento é classificado como anomalia.

## O que foi alterado em relação ao tutorial original

| Original | Aqui |
|---|---|
| `arff2pandas` e `df.append` (removido no pandas 2) | `scipy.io.arff` e `pd.concat` |
| Lote de tamanho 1 (muito lento) | `DataLoader` com lotes de 64 |
| `sns.distplot` (obsoleto) | `sns.histplot` |
| Limiar fixo (`THRESHOLD = 26`) | Limiar escolhido a partir dos dados |

## Roteiro dos desafios

| # | Tarefa |
|---|---|
| 1 | Separar normais e anomalias e dividir em treino/validação/teste |
| 2 | Converter para tensores no formato `(N, tempo, características)` |
| 3 | Completar o `forward` do **Encoder** |
| 4 | Completar o `forward` do **Decoder** |
| 5 | Completar o `forward` do **RecurrentAutoencoder** |
| 6 | Completar o laço de treino |
| 7 | Calcular o erro de reconstrução por amostra |
| 8 | Escolher o limiar |
| 9 | Avaliar com precisão, revocação e F1 |

In [ ]:
import copy
import zipfile
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

sns.set(style="whitegrid", font_scale=1.1)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

## Dados

O conjunto **ECG5000** tem 5.000 séries temporais com 140 pontos cada. Cada série é um batimento cardíaco de um paciente com insuficiência cardíaca congestiva. Há 5 classes:

| Código | Classe |
|---|---|
| 1 | Normal (N) |
| 2 | R-on-T Premature Ventricular Contraction |
| 3 | Premature Ventricular Contraction (PVC) |
| 4 | Supra-ventricular Premature or Ectopic Beat (SP ou EB) |
| 5 | Unclassified Beat (UB) |

Vamos baixar o arquivo (mesmo identificador do notebook original). **Se o download falhar**, baixe o ECG5000 manualmente em <http://timeseriesclassification.com/description.php?Dataset=ECG5000> e coloque os arquivos `ECG5000_TRAIN.arff` e `ECG5000_TEST.arff` na pasta do notebook.

In [ ]:
# Execute apenas se os arquivos .arff ainda não estiverem na pasta
!pip install -q gdown
!gdown --id 16MIleqoIr1vYxlGk4GKnGmrsCPuWkkpT -O ECG5000.zip

with zipfile.ZipFile("ECG5000.zip") as z:
    z.extractall(".")

In [ ]:
def carregar_arff(caminho):
    dados, _ = arff.loadarff(caminho)
    df = pd.DataFrame(dados)
    colunas = list(df.columns)
    colunas[-1] = "target"
    df.columns = colunas
    # a classe vem como bytes (b'1'); convertemos para inteiro
    df["target"] = df["target"].apply(lambda v: int(v.decode() if isinstance(v, bytes) else v))
    return df

train_raw = carregar_arff("ECG5000_TRAIN.arff")
test_raw = carregar_arff("ECG5000_TEST.arff")

df = pd.concat([train_raw, test_raw], ignore_index=True)
df = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)   # embaralha
print(df.shape)

CLASS_NORMAL = 1
class_names = ["Normal", "R on T", "PVC", "SP", "UB"]
df.head()

### Análise exploratória

Distribuição das classes e forma média de cada tipo de batimento:

In [ ]:
contagem = df["target"].value_counts().sort_index()
ax = sns.barplot(x=[class_names[i - 1] for i in contagem.index], y=contagem.values)
ax.set_ylabel("Quantidade")
plt.show()

fig, axs = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
for ax, cls in zip(axs.flat, sorted(df["target"].unique())):
    media = df[df["target"] == cls].drop(columns="target").mean(axis=0).to_numpy()
    ax.plot(media, linewidth=2)
    ax.set_title(class_names[cls - 1])
axs.flat[-1].axis("off")
plt.tight_layout()
plt.show()

## Desafio 1: separar normais e anomalias, e dividir os dados

- `normal_df`: apenas batimentos da classe `CLASS_NORMAL`, **sem** a coluna `target`.
- `anomaly_df`: batimentos de todas as outras classes, **sem** a coluna `target`.
- Divida `normal_df` em **treino (85%)**, **validação (≈ 10%)** e **teste (≈ 5%)** usando `train_test_split` duas vezes: primeiro 85/15, depois divida os 15% em validação e teste com `test_size=0.33`.

*Dica:* use `random_state=RANDOM_SEED`.

In [ ]:
# TODO: crie normal_df e anomaly_df
normal_df = None
anomaly_df = None

# TODO: divida normal_df em train_df, val_df e test_df
train_df, val_df, test_df = None, None, None

In [ ]:
assert normal_df is not None and "target" not in normal_df.columns, "normal_df incorreto"
assert anomaly_df is not None and "target" not in anomaly_df.columns, "anomaly_df incorreto"
assert len(normal_df) + len(anomaly_df) == len(df)
assert abs(len(train_df) / len(normal_df) - 0.85) < 0.01, "treino deve ter ~85% dos normais"
assert len(val_df) + len(test_df) + len(train_df) == len(normal_df)
print("Desafio 1 OK ->", len(train_df), "treino |", len(val_df), "validação |", len(test_df), "teste |", len(anomaly_df), "anomalias")

## Desafio 2: converter para tensores

O LSTM com `batch_first=True` espera entrada no formato **`(N, tempo, características)`**. Nossas séries são univariadas, então o formato é `(N, 140, 1)`.

Complete `create_dataset` para devolver:
- `X`: tensor `float32` de formato `(N, seq_len, n_features)`;
- `seq_len` e `n_features`.

*Dica:* `torch.tensor(df.to_numpy(dtype=np.float32))` tem formato `(N, 140)`. Use `unsqueeze(-1)` para adicionar a dimensão de características.

In [ ]:
def create_dataset(df):
    # TODO: converta o DataFrame em um tensor (N, seq_len, 1)
    X = None
    # TODO: obtenha seq_len e n_features a partir de X
    seq_len, n_features = None, None
    return X, seq_len, n_features

In [ ]:
X_train, seq_len, n_features = create_dataset(train_df)
X_val, _, _ = create_dataset(val_df)
X_test_normal, _, _ = create_dataset(test_df)
X_test_anomaly, _, _ = create_dataset(anomaly_df)

assert X_train.shape[1:] == (140, 1) and X_train.dtype == torch.float32, "formato esperado: (N, 140, 1) float32"
assert seq_len == 140 and n_features == 1
print("Desafio 2 OK ->", tuple(X_train.shape))

BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val), batch_size=BATCH_SIZE)

## O modelo: LSTM Autoencoder

### Teoria

Para séries temporais usamos camadas **LSTM** no lugar das camadas densas, pois elas capturam dependências ao longo do tempo.

- **Encoder:** lê a sequência inteira e resume tudo no **último estado oculto** $h_T$ da LSTM. Esse vetor de dimensão `embedding_dim` é o código latente $z$.
- **Decoder:** recebe $z$, **repete-o** `seq_len` vezes (uma por passo de tempo) e o passa por LSTMs e por uma camada linear para gerar a sequência reconstruída.

```
(N, 140, 1) --LSTM--> --LSTM--> h_T: (N, emb) --repete--> (N, 140, emb) --LSTM--> --LSTM--> --Linear--> (N, 140, 1)
```

Atenção ao que cada chamada `LSTM` devolve: `saida, (h_n, c_n) = lstm(x)`.
- `saida` tem formato `(N, tempo, hidden)`.
- `h_n` tem formato `(num_layers, N, hidden)`. Com 1 camada, `h_n[-1]` é `(N, hidden)`.

## Desafio 3: o Encoder

Complete o `forward`. Passe `x` por `rnn1` e depois por `rnn2`, e devolva o último estado oculto da segunda LSTM, com formato `(N, embedding_dim)`.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, seq_len, n_features, embedding_dim=64):
        super().__init__()
        self.seq_len, self.n_features = seq_len, n_features
        self.embedding_dim, self.hidden_dim = embedding_dim, 2 * embedding_dim

        self.rnn1 = nn.LSTM(input_size=n_features, hidden_size=self.hidden_dim, num_layers=1, batch_first=True)
        self.rnn2 = nn.LSTM(input_size=self.hidden_dim, hidden_size=embedding_dim, num_layers=1, batch_first=True)

    def forward(self, x):
        # x: (N, seq_len, n_features)
        # TODO: passe x por rnn1 e rnn2 e devolva o último estado oculto (N, embedding_dim)
        raise NotImplementedError

In [ ]:
_enc = Encoder(seq_len, n_features, 16)
_z = _enc(torch.randn(5, seq_len, n_features))
assert _z.shape == (5, 16), f"o Encoder deve devolver (N, embedding_dim); obtido {tuple(_z.shape)}"
print("Desafio 3 OK ->", tuple(_z.shape))

## Desafio 4: o Decoder

Complete o `forward`:

1. Transforme `z` de `(N, input_dim)` em `(N, seq_len, input_dim)` repetindo-o ao longo do tempo (`unsqueeze(1)` e `repeat`).
2. Passe por `rnn1` e `rnn2`.
3. Aplique `output_layer` a cada passo de tempo. A saída final deve ter formato `(N, seq_len, n_features)`.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, seq_len, input_dim=64, n_features=1):
        super().__init__()
        self.seq_len, self.input_dim = seq_len, input_dim
        self.hidden_dim, self.n_features = 2 * input_dim, n_features

        self.rnn1 = nn.LSTM(input_size=input_dim, hidden_size=input_dim, num_layers=1, batch_first=True)
        self.rnn2 = nn.LSTM(input_size=input_dim, hidden_size=self.hidden_dim, num_layers=1, batch_first=True)
        self.output_layer = nn.Linear(self.hidden_dim, n_features)

    def forward(self, z):
        # z: (N, input_dim)
        # TODO: repita z ao longo do tempo, passe pelas LSTMs e pela camada linear
        raise NotImplementedError

In [ ]:
_dec = Decoder(seq_len, 16, n_features)
_out = _dec(torch.randn(5, 16))
assert _out.shape == (5, seq_len, n_features), f"o Decoder deve devolver (N, seq_len, n_features); obtido {tuple(_out.shape)}"
print("Desafio 4 OK ->", tuple(_out.shape))

## Desafio 5: juntar tudo

Complete `RecurrentAutoencoder`: o `forward` passa a entrada pelo encoder e depois pelo decoder.

In [ ]:
class RecurrentAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, embedding_dim=64):
        super().__init__()
        self.encoder = Encoder(seq_len, n_features, embedding_dim)
        self.decoder = Decoder(seq_len, embedding_dim, n_features)

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
model = RecurrentAutoencoder(seq_len, n_features, embedding_dim=64).to(device)
_x = X_train[:4].to(device)
assert model(_x).shape == _x.shape, "a saída do autoencoder deve ter o mesmo formato da entrada"
print("Desafio 5 OK")
print("Parâmetros:", sum(p.numel() for p in model.parameters()))

## Desafio 6: laço de treino

Complete o corpo do laço. Para cada lote:

1. Zerar gradientes
2. Passada direta
3. Calcular a perda (alvo = a própria entrada)
4. Retropropagar
5. Atualizar pesos

Usaremos `nn.L1Loss` (erro absoluto médio), que no tutorial original produziu reconstruções melhores que o MSE. A etapa de validação e a seleção do melhor modelo já estão prontas.

In [ ]:
def train_model(model, train_loader, val_loader, n_epochs, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()
    history = {"train": [], "val": []}
    best_wts, best_loss = copy.deepcopy(model.state_dict()), float("inf")

    for epoch in range(1, n_epochs + 1):
        model.train()
        train_losses = []
        for (seq_true,) in train_loader:
            seq_true = seq_true.to(device)
            # TODO: zerar gradientes, passada direta, perda, backward e step
            loss = None
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for (seq_true,) in val_loader:
                seq_true = seq_true.to(device)
                val_losses.append(criterion(model(seq_true), seq_true).item())

        train_loss, val_loss = np.mean(train_losses), np.mean(val_losses)
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        if val_loss < best_loss:
            best_loss, best_wts = val_loss, copy.deepcopy(model.state_dict())
        print(f"Época {epoch:3d} | treino {train_loss:.4f} | validação {val_loss:.4f}")

    model.load_state_dict(best_wts)
    return model.eval(), history

In [ ]:
# Teste rápido (1 época): a perda deve diminuir em relação ao valor inicial
torch.manual_seed(RANDOM_SEED)
_m = RecurrentAutoencoder(seq_len, n_features, 16).to(device)
_, _h = train_model(_m, train_loader, val_loader, n_epochs=2)
assert _h["train"][-1] < _h["train"][0], "a perda de treino não diminuiu: confira o laço"
print("Desafio 6 OK")

Agora treine o modelo de verdade. Em CPU, 30 épocas levam alguns minutos; em GPU, bem menos. Aumente `N_EPOCHS` se quiser melhores resultados (o tutorial original usou 150).

In [ ]:
N_EPOCHS = 30

torch.manual_seed(RANDOM_SEED)
model = RecurrentAutoencoder(seq_len, n_features, embedding_dim=64).to(device)
model, history = train_model(model, train_loader, val_loader, n_epochs=N_EPOCHS)

plt.plot(history["train"], label="treino")
plt.plot(history["val"], label="validação")
plt.xlabel("Época"); plt.ylabel("Erro absoluto médio (L1)")
plt.title("Curva de perda"); plt.legend(); plt.show()

## Desafio 7: erro de reconstrução por amostra

Para detectar anomalias precisamos do erro de **cada** sequência, não da média do lote. Complete `reconstruction_errors`, que recebe um tensor `X` de formato `(N, seq_len, n_features)` e devolve um array NumPy de tamanho `N` com o **erro absoluto médio** de cada sequência (média sobre tempo e características).

*Dica:* `(pred - X).abs().mean(dim=(1, 2))`. Processe em lotes e use `torch.no_grad()`.

In [ ]:
def reconstruction_errors(model, X, batch_size=256):
    model.eval()
    errors = []
    # TODO: percorra X em lotes, reconstrua e calcule o erro absoluto médio por sequência
    return np.concatenate(errors)

In [ ]:
err_train = reconstruction_errors(model, X_train)
err_normal = reconstruction_errors(model, X_test_normal)
err_anomaly = reconstruction_errors(model, X_test_anomaly)

assert err_train.shape == (len(X_train),), "deve haver um erro por sequência"
assert err_anomaly.mean() > err_normal.mean(), "anomalias deveriam ter erro maior que normais; revise o treino"
print("Desafio 7 OK")
print(f"Erro médio -> treino: {err_train.mean():.4f} | normal (teste): {err_normal.mean():.4f} | anomalia: {err_anomaly.mean():.4f}")

sns.histplot(err_normal, bins=40, color="C0", label="normal", stat="density", alpha=0.6)
sns.histplot(err_anomaly, bins=40, color="C3", label="anomalia", stat="density", alpha=0.6)
plt.xlabel("Erro de reconstrução"); plt.legend(); plt.show()

## Desafio 8: escolher o limiar

Olhe o histograma acima. As distribuições se sobrepõem parcialmente: **qualquer limiar** gera algum erro.

Defina `THRESHOLD` como um **percentil** dos erros de treino (apenas dados normais!). Por exemplo, o percentil 95 significa que aceitamos classificar erradamente cerca de 5% dos batimentos normais como anomalia.

Experimente percentis diferentes (90, 95, 99) e observe como o resultado do desafio 9 muda.

*Pergunta de reflexão:* em um sistema de monitoramento cardíaco, é pior um falso positivo ou um falso negativo? Como isso influencia o limiar?

In [ ]:
# TODO: escolha o percentil e calcule o limiar a partir de err_train
PERCENTIL = None
THRESHOLD = None

In [ ]:
assert THRESHOLD is not None, "defina THRESHOLD"
print(f"Limiar (percentil {PERCENTIL}): {THRESHOLD:.4f}")

## Desafio 9: avaliação

Construa o vetor de rótulos verdadeiros (0 = normal, 1 = anomalia) e as predições (1 se erro > limiar). As anomalias são muito mais numerosas que os normais no teste, então **equilibramos** usando apenas as primeiras `len(err_normal)` anomalias, como no tutorial original.

Calcule:
- `y_true`: zeros para normais e uns para anomalias;
- `y_pred`: `1` quando o erro for maior que `THRESHOLD`.

In [ ]:
n = len(err_normal)
errors_all = np.concatenate([err_normal, err_anomaly[:n]])

# TODO: construa y_true e y_pred
y_true = None
y_pred = None

In [ ]:
assert y_true is not None and y_pred is not None, "construa y_true e y_pred"
assert y_true.shape == y_pred.shape == (2 * n,)

print(classification_report(y_true, y_pred, target_names=["normal", "anomalia"], digits=3))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["normal", "anomalia"], yticklabels=["normal", "anomalia"])
plt.xlabel("Predito"); plt.ylabel("Real"); plt.show()

### Visualizando reconstruções

Compare o sinal original com a reconstrução para alguns batimentos normais e anômalos:

In [ ]:
def plot_prediction(x, title, ax):
    with torch.no_grad():
        pred = model(x.unsqueeze(0).to(device)).cpu().squeeze()
    erro = (pred - x.squeeze()).abs().mean().item()
    ax.plot(x.squeeze(), label="original")
    ax.plot(pred, label="reconstruído")
    ax.set_title(f"{title} (erro: {erro:.3f})")

fig, axs = plt.subplots(2, 5, figsize=(20, 7), sharey=True, sharex=True)
for i in range(5):
    plot_prediction(X_test_normal[i], "Normal", axs[0, i])
    plot_prediction(X_test_anomaly[i], "Anomalia", axs[1, i])
axs[0, 0].legend()
plt.tight_layout(); plt.show()

## Desafios extras (opcionais)

1. **Varie o `embedding_dim`** (8, 32, 128) e compare o F1. O que acontece com gargalos muito pequenos ou muito grandes?
2. **Troque L1 por MSE** e compare as reconstruções.
3. **Curva ROC:** calcule o AUC usando `sklearn.metrics.roc_auc_score(y_true, errors_all)`. Esta métrica avalia o modelo **independentemente do limiar**.
4. **Multiclasse:** em vez de agrupar todas as anomalias, avalie o erro médio por classe (R-on-T, PVC, SP, UB). Alguma classe é mais difícil de detectar?
5. **Salvar o modelo:** use `torch.save(model.state_dict(), "ecg_ae.pth")` e recarregue com `load_state_dict`.

## Referências

- Valkov, V. *Time Series Anomaly Detection using LSTM Autoencoders with PyTorch in Python* (Getting Things Done with PyTorch).
- Srivastava, N. et al. *Unsupervised learning of video representations using LSTMs.* ICML, 2015.
- Dataset ECG5000: <http://timeseriesclassification.com/description.php?Dataset=ECG5000>